# 02 — Classic ML Baselines
Handcrafted features + sklearn/xgboost models to establish a baseline.

## Setup

In [9]:
import os, glob, random, warnings, gc
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
from collections import Counter
from dataclasses import dataclass, field
from typing import List
import librosa
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings('ignore')
SEED = 42; random.seed(SEED); np.random.seed(SEED)

@dataclass
class Config:
    data_root: str = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
    output_dir: str = '/kaggle/working'
    sr: int = 22050
    duration: float = 10.0
    n_mfcc: int = 20
    genres: List[str] = field(default_factory=lambda: sorted([
        'blues','classical','country','disco','hiphop',
        'jazz','metal','pop','reggae','rock']))
    stem_types: List[str] = field(default_factory=lambda: ['drums','vocals','bass'])
    entity: str = "23f3003225-indian-institute-of-technology-madras"
    project: str = "23f3003225-dl-genai-project"
    @property
    def stems_dir(self): return os.path.join(self.data_root, 'genres_stems')
    @property
    def test_dir(self): return os.path.join(self.data_root, 'mashups')
    @property
    def test_csv(self): return os.path.join(self.data_root, 'test.csv')

CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)
print("Setup done.")

Setup done.


## WandB

In [10]:
os.system('pip install wandb --upgrade --no-cache-dir -q')
import wandb
wandb.login(key="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")

run = wandb.init(
    entity=CFG.entity, project=CFG.project, name="02_classicml",
    config={"phase": "classic_ml", "n_mfcc": CFG.n_mfcc},
    tags=["classic_ml", "baseline"], job_type="train",
)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


## Feature Extraction
Extract MFCCs, spectral, chroma, tempo features from mixed stems.

In [11]:
def extract_features(y, sr, n_mfcc=20):
    f = {}
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        f[f'mfcc_{i}_mean'] = np.mean(mfccs[i]); f[f'mfcc_{i}_std'] = np.std(mfccs[i])
    delta = librosa.feature.delta(mfccs[:13])
    for i in range(13): f[f'delta_mfcc_{i}_mean'] = np.mean(delta[i])
    for name, fn in [('spectral_centroid', librosa.feature.spectral_centroid),
                     ('spectral_bandwidth', librosa.feature.spectral_bandwidth),
                     ('spectral_rolloff', librosa.feature.spectral_rolloff)]:
        v = fn(y=y, sr=sr)[0]; f[f'{name}_mean'] = np.mean(v); f[f'{name}_std'] = np.std(v)
    sf = librosa.feature.spectral_flatness(y=y)[0]
    f['spectral_flatness_mean'] = np.mean(sf); f['spectral_flatness_std'] = np.std(sf)
    sc = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(sc.shape[0]): f[f'spectral_contrast_{i}_mean'] = np.mean(sc[i])
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i in range(12): f[f'chroma_{i}_mean'] = np.mean(chroma[i])
    try:
        t = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
        for i in range(6): f[f'tonnetz_{i}_mean'] = np.mean(t[i])
    except:
        for i in range(6): f[f'tonnetz_{i}_mean'] = 0.0
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    f['tempo'] = float(tempo) if np.isscalar(tempo) else float(tempo[0])
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    f['zcr_mean'] = np.mean(zcr); f['zcr_std'] = np.std(zcr)
    rms = librosa.feature.rms(y=y)[0]
    f['rms_mean'] = np.mean(rms); f['rms_std'] = np.std(rms)
    return f

In [12]:
# Extract train features
train_feats, train_labels = [], []
feat_names = None

for genre in tqdm(CFG.genres, desc="Train"):
    gp = os.path.join(CFG.stems_dir, genre)
    songs = sorted(s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s)))
    for song in songs:
        stems = []
        for st in CFG.stem_types:
            fp = os.path.join(gp, song, f"{st}.wav")
            if os.path.exists(fp):
                y, sr = librosa.load(fp, sr=CFG.sr, duration=CFG.duration, mono=True)
                if y is not None: stems.append(y)
        if not stems: continue
        mx_len = max(len(s) for s in stems)
        mix = np.zeros(mx_len, dtype=np.float32)
        for s in stems: mix[:len(s)] += s
        mx = np.max(np.abs(mix))
        if mx > 0: mix /= mx
        feats = extract_features(mix, CFG.sr)
        if feat_names is None: feat_names = list(feats.keys())
        train_feats.append(list(feats.values()))
        train_labels.append(genre)

X_train = np.array(train_feats, dtype=np.float32)
y_train = np.array(train_labels)
print(f"X_train: {X_train.shape}, labels: {Counter(y_train)}")

Train:   0%|          | 0/10 [00:00<?, ?it/s]

X_train: (1000, 91), labels: Counter({np.str_('blues'): 100, np.str_('classical'): 100, np.str_('country'): 100, np.str_('disco'): 100, np.str_('hiphop'): 100, np.str_('jazz'): 100, np.str_('metal'): 100, np.str_('pop'): 100, np.str_('reggae'): 100, np.str_('rock'): 100})


In [ ]:
# Extract test features
test_df = pd.read_csv(CFG.test_csv, dtype={'id': str})
test_feats, test_ids = [], []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test"):
    path = None
    for pat in [f"song{str(row['id']).zfill(4)}.wav", f"{row['id']}.wav"]:
        p = os.path.join(CFG.test_dir, pat)
        if os.path.exists(p): path = p; break
    if path is None:
        test_feats.append([0.0]*len(feat_names))
    else:
        y, sr = librosa.load(path, sr=CFG.sr, duration=CFG.duration, mono=True)
        mx = np.max(np.abs(y))
        if mx > 0: y /= mx
        test_feats.append(list(extract_features(y, CFG.sr).values()))
    test_ids.append(row['id'])

X_test = np.array(test_feats, dtype=np.float32)
print(f"X_test: {X_test.shape}")

Test:   0%|          | 0/3020 [00:00<?, ?it/s]

## Model Training & Evaluation
5-fold cross-validation with Macro F1.

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_train)
X_clean = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test_clean = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
X_test_scaled = scaler.transform(X_test_clean)

models = {
    'RandomForest': RandomForestClassifier(n_estimators=500, max_depth=20, random_state=SEED, n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, random_state=SEED),
    'SVM': SVC(C=10, gamma='scale', kernel='rbf', probability=True, random_state=SEED),
    'KNN': KNeighborsClassifier(n_neighbors=7, metric='manhattan', n_jobs=-1),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, random_state=SEED, n_jobs=-1),
}

results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, model in models.items():
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_scaled, y_encoded)):
        X_tr, X_val = X_scaled[tr_idx], X_scaled[val_idx]
        y_tr, y_val = y_encoded[tr_idx], y_encoded[val_idx]
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        f1 = f1_score(y_val, preds, average='macro')
        fold_scores.append(f1)
    mean_f1 = np.mean(fold_scores)
    results[name] = mean_f1
    print(f"{name}: val_f1 = {mean_f1:.4f} (±{np.std(fold_scores):.4f})")
    wandb.log({f"classicml/{name}_f1": mean_f1})

print(f"\nBest: {max(results, key=results.get)} = {max(results.values()):.4f}")

## Confusion Matrix (Best Model)

In [ ]:
# Train best model on full data and show confusion matrix
best_name = max(results, key=results.get)
best_model = models[best_name]

# Use last fold's val set for visualization
best_model.fit(X_scaled[tr_idx], y_encoded[tr_idx])
val_preds = best_model.predict(X_scaled[val_idx])

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_encoded[val_idx], val_preds)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title(f"{best_name} — Confusion Matrix (Last Fold)")
plt.tight_layout()
wandb.log({"plots/confusion_matrix": wandb.Image(fig)}); plt.show()

## Voting Ensemble

In [ ]:
# Soft voting ensemble of top 3
sorted_models = sorted(results.items(), key=lambda x: x[1], reverse=True)[:3]
print(f"Ensemble members: {[m[0] for m in sorted_models]}")

estimators = [(name, models[name]) for name, _ in sorted_models]
ensemble = VotingClassifier(estimators=estimators, voting='soft')

fold_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_scaled, y_encoded)):
    ensemble.fit(X_scaled[tr_idx], y_encoded[tr_idx])
    preds = ensemble.predict(X_scaled[val_idx])
    fold_scores.append(f1_score(y_encoded[val_idx], preds, average='macro'))

ens_f1 = np.mean(fold_scores)
print(f"Ensemble val_f1 = {ens_f1:.4f} (±{np.std(fold_scores):.4f})")
wandb.log({"classicml/ensemble_f1": ens_f1})

## Generate Submission

In [ ]:
# Train ensemble on full data
ensemble.fit(X_scaled, y_encoded)
test_preds = ensemble.predict(X_test_scaled)
test_labels = le.inverse_transform(test_preds)

submission = pd.read_csv(CFG.test_csv, dtype={'id': str})
submission['genre'] = test_labels
submission.to_csv(os.path.join(CFG.output_dir, 'submission.csv'), index=False)
print(submission.head())
print(f"\nPrediction distribution: {Counter(test_labels)}")

## Results Summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
all_results = {**results, 'Ensemble': ens_f1}
names = list(all_results.keys())
scores = list(all_results.values())
colors = ['steelblue']*len(results) + ['darkorange']
ax.barh(names, scores, color=colors)
ax.set_xlabel('Macro F1'); ax.set_title('Classic ML Results')
for i, v in enumerate(scores):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center')
plt.tight_layout()
wandb.log({"plots/model_comparison": wandb.Image(fig)}); plt.show()

wandb.log({"classicml/status": "complete", "classicml/best_f1": max(all_results.values())})
wandb.finish()
print("exp_002_classicml complete.")